# OpenML · Phase 0 smoke test
Confirms the develop container can reach **MinIO** (S3), **Postgres**, **Redis**, and that **torch** works.
Run all cells top to bottom.

In [ ]:
import os, io, boto3
s3 = boto3.client('s3',
    endpoint_url=os.environ['OPENML_S3_ENDPOINT'],
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'])
print('buckets:', [b['Name'] for b in s3.list_buckets()['Buckets']])

In [ ]:
# write an object, then read it back
s3.put_object(Bucket='datasets', Key='smoke/hello.txt', Body=b'hello from OpenML')
obj = s3.get_object(Bucket='datasets', Key='smoke/hello.txt')
print('read back:', obj['Body'].read().decode())

In [ ]:
# Postgres + Redis connectivity
import redis
r = redis.from_url(os.environ['OPENML_REDIS_URL']); r.set('openml:smoke', 'ok')
print('redis:', r.get('openml:smoke').decode())
print('pg dsn:', os.environ['OPENML_PG_DSN'])

In [ ]:
import torch
print('torch', torch.__version__, '| a@b =', (torch.randn(3,3) @ torch.randn(3,3)).shape)
print('NOTE: no Apple GPU (MPS) inside containers — device is CPU here:', torch.device('cpu'))